# linalg-solve-batched composite — cx4: patch singular slices with eye(n) before batched solve

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `linalg-solve-batched`, `singular-matrix-mask-trick`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "linalg-solve-batched"
DD_ATOM_IDS = ["linalg-solve-batched", "singular-matrix-mask-trick"]
DD_SUBTOPICS = ["PyTorch: Batched linalg.solve", "Numpy: Singular matrix mask trick"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

`t.linalg.solve(A, b)` crashes the ENTIRE batched call if even one slice is singular — there's no per-slice error mode. The `singular-matrix-mask-trick` is the standard workaround: detect singular slices via `|det| < eps`, overwrite them with the identity (so the solve succeeds on garbage data), then mask the spurious outputs afterwards.

The composition is THE protective shell around batched solve:

1. `dets = t.linalg.det(A)` → `(K,)`
2. `is_singular = dets.abs() < eps` → `(K,)` bool
3. `A_safe = A.clone()`; `A_safe[is_singular] = t.eye(n)` (broadcasts I across the singular slices)
4. `x = t.linalg.solve(A_safe, b)` — never crashes now
5. Return `(x, ~is_singular)` so the caller knows which rows are real.

This is the production pattern for any batched solve over data of unknown quality.

### Composite Exercise — patch singular slices with eye(n) before batched solve

**Atoms exercised together**: `linalg-solve-batched`, `singular-matrix-mask-trick`

Implement `cx4_robust_batched_solve(A, b, eps=1e-8)` that runs a batched solve over `(K, n, n)` systems where SOME slices may be singular, without crashing.

Inputs:
- `A`: `(K, n, n)` — coefficient matrices (some may be singular).
- `b`: `(K, n)` — right-hand sides.
- `eps`: float — detection threshold on `|det|`.

Returns `(x, is_valid)`:
- `x: (K, n)` — solutions. Entries at singular slices are garbage (don't trust them).
- `is_valid: (K,) bool` — `True` where the original `A[k]` was non-singular.

**Steps:**
1. `dets = t.linalg.det(A)`, `is_singular = dets.abs() < eps`.
2. `A_safe = A.clone()` — do NOT mutate the caller's tensor. Then `A_safe[is_singular] = t.eye(n)` to overwrite singular slices.
3. `x = t.linalg.solve(A_safe, b)` — single batched call, no loop.
4. Return `x, ~is_singular`.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx4_robust_batched_solve(A, b, eps=1e-8):
    raise NotImplementedError

def _test_cx4():
    # Case A: hand-picked 3x3 batch with one singular slice in the middle.
    A = t.stack([
        t.tensor([[2.0, 0.0, 0.0], [0.0, 2.0, 0.0], [0.0, 0.0, 2.0]]),   # det=8, x = b/2
        t.tensor([[1.0, 2.0, 3.0], [2.0, 4.0, 6.0], [0.0, 0.0, 1.0]]),   # SINGULAR (row1 = 2*row0)
        t.tensor([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]]),   # identity, x = b
    ])
    b = t.tensor([
        [4.0, 6.0, 8.0],
        [1.0, 2.0, 3.0],
        [-1.0, 0.5, 2.0],
    ])
    A_before = A.clone()
    x, is_valid = cx4_robust_batched_solve(A, b)

    # Did NOT mutate input.
    assert t.equal(A, A_before), 'must not mutate the input A in place'
    # Shapes / dtypes.
    assert tuple(x.shape) == (3, 3), f'x shape: {tuple(x.shape)}'
    assert tuple(is_valid.shape) == (3,)
    assert is_valid.dtype == t.bool
    # Mask correctness.
    assert is_valid.tolist() == [True, False, True], f'is_valid: {is_valid.tolist()}'
    # Non-singular solutions are correct.
    assert t.allclose(x[0], t.tensor([2.0, 3.0, 4.0]), atol=1e-5), f'x[0]: {x[0]}'
    assert t.allclose(x[2], t.tensor([-1.0, 0.5, 2.0]), atol=1e-5), f'x[2]: {x[2]}'
    # x[1] is undefined, but must be finite (no NaN propagation).
    assert t.isfinite(x[1]).all(), f'x[1] must be finite (identity gives finite garbage): {x[1]}'

    # Case B: all singular — confirm no crash, all is_valid=False.
    A_all_sing = t.zeros(3, 2, 2)
    b_s = t.randn(3, 2)
    x_s, valid_s = cx4_robust_batched_solve(A_all_sing, b_s)
    assert tuple(x_s.shape) == (3, 2)
    assert not valid_s.any(), 'all should be invalid'
    assert t.isfinite(x_s).all(), 'no NaN/inf even when all singular'

    # Case C: all non-singular — cross-check against plain batched solve.
    rng = t.Generator().manual_seed(4)
    K, n = 16, 3
    A_good = t.eye(n).unsqueeze(0).expand(K, n, n) + 0.5 * t.randn(K, n, n, generator=rng)
    b_g = t.randn(K, n, generator=rng)
    x_g, valid_g = cx4_robust_batched_solve(A_good, b_g)
    assert valid_g.all(), 'all should be valid (well-conditioned)'
    assert t.allclose(x_g, t.linalg.solve(A_good, b_g), atol=1e-4)
    _dd_passed.add('cx4')

_test_cx4()

<details><summary>Show solution — cx4</summary>

```python
def cx4_robust_batched_solve(A, b, eps=1e-8):
    K, n, _ = A.shape
    # singular-matrix-mask-trick: detect via determinant.
    dets = t.linalg.det(A)
    is_singular = dets.abs() < eps
    # Clone so we don't mutate the caller's tensor.
    A_safe = A.clone()
    # Overwrite singular slices with the identity — broadcast (n,n) eye into the
    # boolean-indexed (M, n, n) slice.
    A_safe[is_singular] = t.eye(n)
    # linalg-solve-batched: never crashes now, but garbage at singular slices.
    x = t.linalg.solve(A_safe, b)
    return x, ~is_singular
```

The two atoms are tightly coupled in real ARENA code — you basically never use raw `linalg.solve` on data of unknown provenance. The mask trick is the standard armor.

Why identity specifically: any non-singular matrix would let the solve succeed, but `eye(n)` produces solutions of `x = b` at those slices — i.e. FINITE garbage. If a downstream consumer ever forgets to mask via `is_valid`, they'll see weird answers but never `NaN` propagation, which makes the bug debuggable instead of silent.

The det threshold (1e-8) is appropriate for float32; for float64 use 1e-12. A more principled approach uses condition number (`t.linalg.cond`) but `|det| < eps` is the ARENA convention and is much faster.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx4'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx4',
        'subtopics': ["PyTorch: Batched linalg.solve", "Numpy: Singular matrix mask trick"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()